<a href="https://colab.research.google.com/github/aahan-charak24/Deep-Learning-Mastery/blob/main/Codes/ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import DataLoader
import torch.optim as optim

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
print(device)

cuda


<h2>DataLoader</h2>

In [4]:
def data_loader(data_dir, batch_size, random_seed = 42, valid_size = 0.1, shuffle = True, test = False):

  #normalize the images
  normalize = transforms.Normalize(
       mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010],
  )

  #transforms
  transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      normalize,
  ])

  #check whether the dataset is test
  if test:
    dataset = datasets.CIFAR10(
        root = data_dir, train = False, download = True, transform = transform
    )

    #dataloader for test
    dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

    return dataloader

  #training set
  train_dataset = datasets.CIFAR10(
      root = data_dir, train = True, download = True, transform = transform
  )

  #validation set
  valid_dataset = datasets.CIFAR10(
      root = data_dir, train = True, download = True, transform = transform
  )

  num_train = len(train_dataset)
  indices = list(range(num_train))
  split = int(np.floor(valid_size * num_train))

  #check if shuffle true
  if shuffle:
    np.random.seed(42)
    #shuffle the indices
    np.random.shuffle(indices)

  #now we have random incides in train and valid
  train_idx, valid_idx = indices[split: ], indices[:split]
  #sample randomly from these indices
  #training loader should only use train_idx
  train_sampler = SubsetRandomSampler(train_idx)
  #valid loader should only use valid_idx
  valid_sampler = SubsetRandomSampler(valid_idx)

  #dataloaders
  train_loader = DataLoader(train_dataset, batch_size = batch_size, sampler = train_sampler)
  valid_loader = DataLoader(valid_dataset, batch_size = batch_size, sampler = valid_sampler)

  return (train_loader, valid_loader)







In [5]:
train_loader, valid_loader = data_loader(data_dir = './data', batch_size = 64)


100%|██████████| 170M/170M [13:35<00:00, 209kB/s] 


<h1>ResNet 34 architecture </h1>

<h1>Residual Block<h1>

In [8]:
class ResidualBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
    super(ResidualBlock, self).__init__()

    #first conv
    self.conv1 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding  =1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU()
    )

    #second conv
    self.conv2 = nn.Sequential(
        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
        nn.ReLU()
    )

    self.downsample = downsample
    self.relu = nn.ReLU()
    self.out_channels = out_channels


  #forward pass
  def forward(self, X):
    residual = X
    out = self.conv1(X)
    out = self.conv2(out)

    #if we require residual connection if downsampling required
    if self.downsample:
      residual = self.downsample(X)

    #skip connection addition
    out = out + residual
    out = self.relu(out)

    return out



<h1>Main architecture </h1>

In [9]:
class ResNet(nn.Module):
  def __init__(self, block, layers, num_classes = 10):
    '''
    Layers is how many residual blocks should be present
    '''
    super(ResNet, self).__init__()

    #incoming
    self.inplanes = 64

    self.conv1 = nn.Sequential(
                            nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
                            nn.BatchNorm2d(64),
                            nn.ReLU())

    self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)

    self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
    self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
    self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
    self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)
    self.avgpool = nn.AvgPool2d(7, stride = 1)
    self.fc = nn.Linear(512, num_classes)



  #make layer function
  def _make_layer(self, block, planes, blocks, stride = 1):
    '''
    Block -> how many residual blocks we have
    Stride -> whether to reduce image size
    Planes -> output channel size for the block

    '''
    downsample = None

    #if we are downsampling so input size != output size, we have logic to make them same here

    if stride !=1 or self.inplanes != planes:
      downsample = nn.Sequential(
          nn.Conv2d(self.inplanes, planes, kernel_size = 1, stride = stride),
          nn.BatchNorm2d(planes)
      )

    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes

    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))


    return nn.Sequential(*layers)



  #forward pass
  def forward(self, X):
    x = self.conv1(X)
    x = self.maxpool(x)
    x = self.layer0(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)

    x = self.avgpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)

    return x




<h1>Training Parameters </h1>

In [10]:
model = ResNet(ResidualBlock, [3, 4, 6, 3]).to(device)
print(model)


ResNet(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer0): Sequential(
    (0): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
      (conv2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
      )
      (relu): ReLU()
    )
    (1): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
      (conv

In [11]:
num_classes = 10
num_epochs = 25
batch_size = 32
learning_rate = 0.01


#loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

total_steps = len(train_loader)

In [12]:
import tqdm
import gc

In [13]:
for epoch in range(num_epochs):

    model.train()

    train_bar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, labels in train_bar:
        #move to device
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        #free memory
        del images, labels, outputs

        torch.cuda.empty_cache()

        gc.collect()

        train_bar.set_postfix(loss=loss.item())


    # validation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

            del images, labels, outputs

    accuracy = 100 * correct / total

    print(f"Validation Accuracy: {accuracy:.2f}%")

Epoch 1/25: 100%|██████████| 704/704 [06:51<00:00,  1.71it/s, loss=1.75]


Validation Accuracy: 28.98%


Epoch 2/25: 100%|██████████| 704/704 [06:53<00:00,  1.70it/s, loss=1.4]


Validation Accuracy: 41.56%


Epoch 3/25: 100%|██████████| 704/704 [06:53<00:00,  1.70it/s, loss=2.29]


...

Validation Accuracy: 81.04%


Epoch 24/25: 100%|██████████| 704/704 [06:55<00:00,  1.69it/s, loss=0.394]


Validation Accuracy: 84.02%


Epoch 25/25: 100%|██████████| 704/704 [06:54<00:00,  1.70it/s, loss=0.134]


Validation Accuracy: 82.02%


In [14]:
test_loader = data_loader(data_dir = "./data", batch_size = 64, shuffle = True, test = True)

In [15]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total

  print(f"Test Accuracy: {accuracy: .2f}%")


Test Accuracy:  81.51%
